# Module 4: Comparison and Evaluation

**Duration:** ~25 minutes

**Environment:** AWS g4dn.4xlarge (or local GPU)

## Learning Objectives
- Compare responses from all three approaches
- Apply a structured evaluation rubric
- Visualize performance differences
- Understand when to use each approach

## Overview
Now we'll bring together the results from all three approaches (baseline, RAG, and fine-tuned) and evaluate them systematically.

## Prerequisites
- Completed Modules 1, 2, and 3
- Results files exist in the `results/` directory

## 4.1 Set Working Directory

In [ ]:
import os

def find_project_root():
    """Find the project root directory by looking for key files."""
    current = os.getcwd()
    
    if os.path.exists(os.path.join(current, 'data', 'f5_docs')):
        return current
    
    parent = os.path.dirname(current)
    grandparent = os.path.dirname(parent)
    
    if os.path.exists(os.path.join(grandparent, 'data', 'f5_docs')):
        return grandparent
    
    home = os.path.expanduser('~')
    common_paths = [
        os.path.join(home, 'llm-finetuning-rag-lab'),
        os.path.join(home, 'LLM-FineTuning-RAG-Lab'),
        '/root/llm-finetuning-rag-lab',
    ]
    
    for path in common_paths:
        if os.path.exists(os.path.join(path, 'data', 'f5_docs')):
            return path
    
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
    print(f"Working directory set to: {PROJECT_ROOT}")
else:
    print("ERROR: Could not find project root directory!")
    print("Please run: cd ~/llm-finetuning-rag-lab && source venv/bin/activate && jupyter lab")
    raise FileNotFoundError("Project root not found.")

## 4.2 Load All Results

In [ ]:
import json

print(f"Working directory: {os.getcwd()}")
print(f"\nResults files:")
if os.path.exists('results'):
    for f in os.listdir('results'):
        print(f"  - {f}")
else:
    print("  No results directory - run previous modules first")

In [ ]:
# Load all results
def load_results(filepath):
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Not found: {filepath}")
        return None

baseline_data = load_results('results/baseline_results.json')
rag_data = load_results('results/rag_results.json')
finetuned_data = load_results('results/finetuned_results.json')

# Helper function to format status
def format_status(name, data):
    if data:
        count = len(data.get('responses', []))
        return f"  {name}: Loaded ({count} responses)"
    else:
        return f"  {name}: Missing"

# Check what we have
print("\nResults status:")
print(format_status("Baseline  ", baseline_data))
print(format_status("RAG       ", rag_data))
print(format_status("Fine-tuned", finetuned_data))

if not all([baseline_data, rag_data, finetuned_data]):
    print("\nSome results are missing. Run the previous modules to generate them.")

## 4.3 Side-by-Side Comparison

Let's view responses from all three approaches for each question.

In [ ]:
def display_comparison(question_idx=0):
    """Display responses from all three approaches for a question."""
    print("=" * 80)
    print(f"QUESTION {question_idx + 1}:")
    print(baseline_data['responses'][question_idx]['question'])
    print("=" * 80)
    
    print("\n" + "-" * 40)
    print("BASELINE RESPONSE:")
    print("-" * 40)
    print(baseline_data['responses'][question_idx]['response'])
    
    print("\n" + "-" * 40)
    print("RAG RESPONSE:")
    print("-" * 40)
    print(rag_data['responses'][question_idx]['response'])
    if 'sources' in rag_data['responses'][question_idx]:
        print(f"\nSources: {', '.join(rag_data['responses'][question_idx]['sources'])}")
    
    print("\n" + "-" * 40)
    print("FINE-TUNED RESPONSE:")
    print("-" * 40)
    print(finetuned_data['responses'][question_idx]['response'])

# Display first comparison
display_comparison(0)

In [ ]:
# Display all comparisons
for i in range(len(baseline_data['responses'])):
    display_comparison(i)
    print("\n")

## 4.4 Evaluation Rubric

We'll score each response on 5 criteria (0-5 scale each):

| Criterion | Description |
|-----------|-------------|
| **Accuracy** | Factual correctness of technical information |
| **Completeness** | How fully the question is answered |
| **Specificity** | Inclusion of F5-specific details (TMSH, iRules, etc.) |
| **Actionability** | Practical usefulness - can someone act on this? |
| **Clarity** | Organization and understandability |

### Scoring Guide

**0** - Completely wrong/missing  
**1** - Mostly incorrect  
**2** - Partially correct  
**3** - Reasonably good  
**4** - Good with minor issues  
**5** - Excellent

In [ ]:
# Create evaluation structure
criteria = ['accuracy', 'completeness', 'specificity', 'actionability', 'clarity']

# Example scores (modify based on your observations)
example_scores = {
    'baseline': [
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3},
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 1, 'clarity': 3},
        {'accuracy': 1, 'completeness': 1, 'specificity': 0, 'actionability': 1, 'clarity': 2},
        {'accuracy': 3, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3},
        {'accuracy': 2, 'completeness': 2, 'specificity': 1, 'actionability': 2, 'clarity': 3}
    ],
    'rag': [
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 3, 'specificity': 4, 'actionability': 4, 'clarity': 3},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 3, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4}
    ],
    'finetuned': [
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4},
        {'accuracy': 5, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 4, 'actionability': 4, 'clarity': 4},
        {'accuracy': 4, 'completeness': 4, 'specificity': 5, 'actionability': 5, 'clarity': 4}
    ]
}

print("Example evaluation scores loaded")
print("\nNote: Modify these scores based on your actual observations.")

## 4.5 Calculate Summary Statistics

In [ ]:
import numpy as np

def calculate_stats(scores):
    """Calculate average scores for each criterion and overall."""
    stats = {criterion: [] for criterion in criteria}
    
    for score_set in scores:
        for criterion in criteria:
            stats[criterion].append(score_set[criterion])
    
    averages = {criterion: np.mean(values) for criterion, values in stats.items()}
    averages['total'] = sum(averages.values())
    averages['average'] = averages['total'] / len(criteria)
    
    return averages

baseline_stats = calculate_stats(example_scores['baseline'])
rag_stats = calculate_stats(example_scores['rag'])
finetuned_stats = calculate_stats(example_scores['finetuned'])

print("Summary Statistics\n")
print(f"{'Criterion':<15} {'Baseline':>10} {'RAG':>10} {'Fine-tuned':>12}")
print("-" * 50)
for criterion in criteria:
    print(f"{criterion.title():<15} {baseline_stats[criterion]:>10.2f} {rag_stats[criterion]:>10.2f} {finetuned_stats[criterion]:>12.2f}")
print("-" * 50)
print(f"{'Average':<15} {baseline_stats['average']:>10.2f} {rag_stats['average']:>10.2f} {finetuned_stats['average']:>12.2f}")
print(f"{'Total (/25)':<15} {baseline_stats['total']:>10.2f} {rag_stats['total']:>10.2f} {finetuned_stats['total']:>12.2f}")

## 4.6 Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Grouped bar chart by criteria
x = np.arange(len(criteria))
width = 0.25

ax1 = axes[0]
bars1 = ax1.bar(x - width, [baseline_stats[c] for c in criteria], width, label='Baseline', color='#ff6b6b')
bars2 = ax1.bar(x, [rag_stats[c] for c in criteria], width, label='RAG', color='#4ecdc4')
bars3 = ax1.bar(x + width, [finetuned_stats[c] for c in criteria], width, label='Fine-tuned', color='#45b7d1')

ax1.set_xlabel('Evaluation Criteria', fontsize=12)
ax1.set_ylabel('Score (0-5)', fontsize=12)
ax1.set_title('Performance by Criteria', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([c.title() for c in criteria], rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, 5.5)
ax1.axhline(y=3, color='gray', linestyle='--', alpha=0.5)

# Chart 2: Overall average comparison
ax2 = axes[1]
approaches = ['Baseline', 'RAG', 'Fine-tuned']
averages = [baseline_stats['average'], rag_stats['average'], finetuned_stats['average']]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']

bars = ax2.bar(approaches, averages, color=colors)

ax2.set_xlabel('Approach', fontsize=12)
ax2.set_ylabel('Average Score (0-5)', fontsize=12)
ax2.set_title('Overall Performance Comparison', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 5.5)
ax2.axhline(y=3, color='gray', linestyle='--', alpha=0.5)

# Add value labels
for bar, val in zip(bars, averages):
    ax2.annotate(f'{val:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5),
                textcoords='offset points',
                ha='center', va='bottom',
                fontweight='bold', fontsize=12)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart saved to results/comparison_chart.png")

In [ ]:
# Radar chart for detailed comparison
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Prepare data
angles = np.linspace(0, 2 * np.pi, len(criteria), endpoint=False).tolist()
angles += angles[:1]

baseline_values = [baseline_stats[c] for c in criteria] + [baseline_stats[criteria[0]]]
rag_values = [rag_stats[c] for c in criteria] + [rag_stats[criteria[0]]]
finetuned_values = [finetuned_stats[c] for c in criteria] + [finetuned_stats[criteria[0]]]

ax.plot(angles, baseline_values, 'o-', linewidth=2, label='Baseline', color='#ff6b6b')
ax.fill(angles, baseline_values, alpha=0.1, color='#ff6b6b')

ax.plot(angles, rag_values, 'o-', linewidth=2, label='RAG', color='#4ecdc4')
ax.fill(angles, rag_values, alpha=0.1, color='#4ecdc4')

ax.plot(angles, finetuned_values, 'o-', linewidth=2, label='Fine-tuned', color='#45b7d1')
ax.fill(angles, finetuned_values, alpha=0.1, color='#45b7d1')

ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.title() for c in criteria], fontsize=11)
ax.set_ylim(0, 5)
ax.set_title('Multi-dimensional Performance Comparison', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('results/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRadar chart saved to results/radar_chart.png")

## 4.7 Analysis and Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

# Calculate improvements
rag_improvement = ((rag_stats['average'] - baseline_stats['average']) / baseline_stats['average']) * 100
ft_improvement = ((finetuned_stats['average'] - baseline_stats['average']) / baseline_stats['average']) * 100

print(f"\nImprovement over Baseline:")
print(f"  RAG: +{rag_improvement:.1f}%")
print(f"  Fine-tuned: +{ft_improvement:.1f}%")

# Find best approach per criterion
print(f"\nBest Approach by Criterion:")
for criterion in criteria:
    scores = {
        'Baseline': baseline_stats[criterion],
        'RAG': rag_stats[criterion],
        'Fine-tuned': finetuned_stats[criterion]
    }
    best = max(scores, key=scores.get)
    print(f"  {criterion.title()}: {best} ({scores[best]:.2f})")

# Overall winner
overall_scores = {
    'Baseline': baseline_stats['average'],
    'RAG': rag_stats['average'],
    'Fine-tuned': finetuned_stats['average']
}
winner = max(overall_scores, key=overall_scores.get)
print(f"\nOverall Best: {winner} (avg: {overall_scores[winner]:.2f}/5)")

## 4.8 When to Use Each Approach

In [ ]:
recommendations = """
==================================================================
                APPROACH RECOMMENDATIONS                         
==================================================================

BASELINE (General LLM)
------------------------------------------------------------------
  + General knowledge questions                                   
  + Quick prototyping                                             
  + When no domain data available                                 
  - Poor for domain-specific details                              

RAG (Retrieval-Augmented Generation)                         
------------------------------------------------------------------
  + Factual queries needing source verification                   
  + Frequently updated knowledge bases                            
  + When source attribution is important                          
  + Limited compute/training resources                            
  - Depends on retrieval quality                                  

FINE-TUNED                                                   
------------------------------------------------------------------
  + Consistent domain terminology needed                          
  + Reasoning and pattern-based questions                         
  + Fast inference without retrieval latency                      
  + When training data is available                               
  - Harder to update; risk of overfitting                         

PRODUCTION RECOMMENDATION: COMBINE APPROACHES                
------------------------------------------------------------------
  * Fine-tune for domain understanding and terminology            
  * Add RAG for current/detailed reference information            
  * Implement fallback strategies for edge cases                  
  * Monitor and continuously evaluate performance                 

==================================================================
"""

print(recommendations)

## 4.9 Save Evaluation Report

In [ ]:
from datetime import datetime

# Compile full evaluation report
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "summary": {
        "baseline": {
            "average_score": baseline_stats['average'],
            "total_score": baseline_stats['total'],
            "by_criterion": {c: baseline_stats[c] for c in criteria}
        },
        "rag": {
            "average_score": rag_stats['average'],
            "total_score": rag_stats['total'],
            "by_criterion": {c: rag_stats[c] for c in criteria},
            "improvement_over_baseline": rag_improvement
        },
        "finetuned": {
            "average_score": finetuned_stats['average'],
            "total_score": finetuned_stats['total'],
            "by_criterion": {c: finetuned_stats[c] for c in criteria},
            "improvement_over_baseline": ft_improvement
        }
    },
    "winner": winner,
    "detailed_scores": example_scores,
    "questions": [r['question'] for r in baseline_data['responses']]
}

with open('results/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("Evaluation report saved to results/evaluation_report.json")

In [ ]:
# Generate text report
text_report = f"""
================================================================================
F5 AI TECHNICAL ASSISTANT - EVALUATION REPORT
================================================================================

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Base Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0

--------------------------------------------------------------------------------
SUMMARY SCORES (Scale: 0-5)
--------------------------------------------------------------------------------

Approach        Accuracy  Completeness  Specificity  Actionability  Clarity  AVG
--------------------------------------------------------------------------------
Baseline        {baseline_stats['accuracy']:.2f}      {baseline_stats['completeness']:.2f}          {baseline_stats['specificity']:.2f}          {baseline_stats['actionability']:.2f}          {baseline_stats['clarity']:.2f}     {baseline_stats['average']:.2f}
RAG             {rag_stats['accuracy']:.2f}      {rag_stats['completeness']:.2f}          {rag_stats['specificity']:.2f}          {rag_stats['actionability']:.2f}          {rag_stats['clarity']:.2f}     {rag_stats['average']:.2f}
Fine-tuned      {finetuned_stats['accuracy']:.2f}      {finetuned_stats['completeness']:.2f}          {finetuned_stats['specificity']:.2f}          {finetuned_stats['actionability']:.2f}          {finetuned_stats['clarity']:.2f}     {finetuned_stats['average']:.2f}

--------------------------------------------------------------------------------
IMPROVEMENT OVER BASELINE
--------------------------------------------------------------------------------

RAG:        +{rag_improvement:.1f}%
Fine-tuned: +{ft_improvement:.1f}%

--------------------------------------------------------------------------------
OVERALL WINNER: {winner.upper()}
--------------------------------------------------------------------------------

KEY TAKEAWAYS:
* Both RAG and fine-tuning significantly improve F5-specific responses
* Fine-tuning excels at terminology and actionable guidance
* RAG provides reliable source-backed information
* For production: combine both approaches for best results

================================================================================
"""

with open('results/evaluation_report.txt', 'w') as f:
    f.write(text_report)

print(text_report)
print("Text report saved to results/evaluation_report.txt")

## 4.10 Summary

In this module, we:

1. Loaded and compared all three approaches
2. Applied a structured evaluation rubric
3. Created visualizations of performance
4. Analyzed strengths and weaknesses
5. Generated comprehensive reports

### Key Takeaways

1. **Baseline limitations**: General LLMs lack domain expertise
2. **RAG benefits**: Brings external knowledge without retraining
3. **Fine-tuning benefits**: Internalizes domain patterns and terminology
4. **Best practice**: Combine approaches for production systems

### What's Next?

For production deployment, consider:
- Expanding training data (more Q&A pairs)
- Adding more documentation to RAG
- Implementing hybrid RAG + fine-tuned approach
- Setting up monitoring and evaluation pipelines
- Testing with real users and collecting feedback

## Congratulations!

You've completed the F5 AI Technical Assistant Training Lab!

You've learned how to:
- Load and use quantized LLMs
- Build RAG systems with LangChain and ChromaDB
- Fine-tune models with QLoRA and Unsloth
- Evaluate and compare different approaches

These skills apply broadly to building domain-specific AI assistants for any technical field!